# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadM434-prog/Flyrank-ML-Internship-Assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## ML Task Type: Ranking

My chosen lane is a **ranking** task. The goal is to prioritize webpages based on their refresh opportunity so that the content team can review the highest-impact pages first. Rather than making a simple yes/no prediction, the model produces an ordered list of pages using signals such as search performance, engagement metrics, content freshness, and traffic trends.

A ranking approach is appropriate because editor time is limited. The content team gains more value from knowing which pages to review first than from treating every page equally. The model supports prioritization, while the final editorial decision remains with the content team.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or Proxy

The model will predict the **refresh priority** of each webpage, producing a ranked list of pages for editorial review. The dataset does not contain a direct label indicating which pages should be refreshed first, so the ranking will be based on a **proxy target** rather than an observed outcome.

The proxy is derived from existing performance and freshness indicators in the dataset, including `trend_direction`, `trend_pct`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, and engagement metrics. Pages showing declining performance, older content, or reduced engagement will receive higher refresh priority because they are more likely to benefit from editorial review.

This proxy supports the business decision of prioritizing limited editor time while remaining transparent about the fact that it is a defined rule rather than a directly observed label.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric

The primary success metric is **Precision@K**, which measures how many of the top-ranked pages are genuinely high-priority refresh opportunities according to the proxy target. This aligns with the real workflow because editors typically review only a limited number of pages at a time, making the quality of the highest-ranked recommendations more important than the overall ranking.

As the project becomes more advanced, ranking metrics such as **NDCG (Normalized Discounted Cumulative Gain)** could also be used to evaluate how well the model orders pages by refresh priority. From a business perspective, success means that the model consistently places the most valuable review opportunities near the top of the queue, allowing editors to spend their time where it has the greatest impact.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis

The unit of analysis is **one content page**. Each row in the dataset represents a single webpage and includes information about its search performance, traffic, engagement, freshness, and content characteristics. The ranking model evaluates each page independently and assigns it a refresh priority relative to the other pages in the dataset.

In [3]:
from pathlib import Path
import pandas as pd

root = Path.cwd()

while not (root / "data/raw/content_refresh_anonymized.csv").exists():
    if root.parent == root:
        raise FileNotFoundError("Dataset not found.")
    root = root.parent

df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
ColumnsToShow = [
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct"
]

df[ColumnsToShow].head()

,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,187,20,3803,29,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,7,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,11,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,463,22,11751,58,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,24,0.13,44.0,down,-34.7


In [ ]:
ExampleDf = df[ColumnsToShow].copy()
ExampleDf["refresh_priority"] = 0
ExampleDf.loc[0,"refresh_priority"] = 2
ExampleDf.loc[1,"refresh_priority"] = 1
ExampleDf.loc[2,"refresh_priority"] = 4
ExampleDf.loc[3,"refresh_priority"] = 3
ExampleDf.loc[4,"refresh_priority"] = 5

#The dataset does not contain a `refresh_priority` column. The example above illustrates what the target would look like after defining a proxy label. Lower numbers indicate higher editorial priority, with 1 representing the page that should be reviewed first.
ExampleDf.head()

,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct,refresh_priority
0,content_304f48230142,client_f369cb89fc,187,20,3803,29,0.76,10.6,down,-41.4,2
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,7,0.05,20.3,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,11,0.09,36.5,down,-60.9,4
3,content_331d6c4de07b,client_19581e27de,463,22,11751,58,0.49,6.2,stable,-13.8,3
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,24,0.13,44.0,down,-34.7,5


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats a Fixed Rule

A simple rule-based system can identify obvious cases, such as pages that have not been updated for more than 90 days or pages with declining traffic. However, content refresh decisions depend on multiple interacting factors rather than a single threshold.

For example, a page with declining traffic but only a few impressions is usually less urgent than a high-traffic page with the same decline. Similarly, an older page that still has strong engagement and a good search position may not need immediate attention. Metrics such as `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `content_age_days`, and `trend_pct` all contribute to determining refresh priority.

A ranking model can learn how these signals work together and produce a prioritized review queue instead of relying on rigid rules. This helps editors focus on the pages most likely to benefit from review while remaining flexible as content patterns change over time. The final editorial decision still belongs to the content team; the model supports prioritization rather than replacing human judgment.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.